# SPU demo3
干掉耗时的topk采样
## (1) Load Model & Weights from HuggingFace

In [1]:
import jax

from helper import load_from_cache, generate, generate_ex

base_path="/root/.cache/huggingface/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
model, params, tokenizer = load_from_cache(base_path)

print("model loaded")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


model loaded


In [2]:
def generate_demo(prompt, gen_len=10, seed=42):
    input_ids = tokenizer.encode(prompt, return_tensors='jax')
    output_ids = generate(model, params, input_ids, gen_len, seed=seed)
    print(prompt, tokenizer.decode(output_ids[0]), sep='')

In [3]:
generate_demo('Python is')

Python is in a sense a kind of 'virtual' language


## (2) SPU

### 2.1: 定义simulator

> 教程视频中用了一堆的`spu_pb2`，但现在的spu中根本没有这个模块，根据成员推测，尝试换成`libspu`

In [4]:
import spu.utils.simulation as spsim
import spu.libspu as libspu

# sim_che = spsim.Simulator.simple(2,libspu.ProtocolKind.CHEETAH, libspu.FieldType.FM128)
# sim_aby = spsim.Simulator.simple(3,libspu.ProtocolKind.ABY3, libspu.FieldType.FM128)

下面的参数来自flax_resnet示例，精度很高，输出很理想，但可能运行会慢一些

我们特意让aby的参数和`3pc.json`一致

In [5]:
# define cheetah config with pphlo trace and profile on
config_che = libspu.RuntimeConfig(
    protocol = libspu.ProtocolKind.CHEETAH,
    field = libspu.FieldType.FM128,
    fxp_fraction_bits = 36,
)

config_che.enable_pphlo_profile = True
config_che.enable_hal_profile = True

config_aby = libspu.RuntimeConfig(
    protocol = libspu.ProtocolKind.ABY3,
    field = libspu.FieldType.FM128,
    fxp_fraction_bits = 36,
)

config_aby.enable_pphlo_profile = True
config_aby.enable_hal_profile = True
config_aby.fxp_exp_mode = libspu.RuntimeConfig.ExpMode.EXP_PADE
config_aby.fxp_div_goldschmidt_iters = 3

sim = spsim.Simulator(3,config_aby)

### 2.2: 定义运行函数

In [6]:
# 目的是加上jit
# 重要：topk非常低效
@jax.jit
def gen_spu(params, input_ids):
    # return generate(model, params, input_ids, n_tokens_to_gen=3) # 默认topk
    return generate_ex(model, params, input_ids, n_tokens_to_gen=3) # 无topk

### 2.3: 运行密态程序

In [7]:
# plaintext test
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
output_ids = gen_spu(params, input_ids)
print(prompt, tokenizer.decode(output_ids[0]), sep='')

Python is a great tool


In [8]:
# SPU emulation mode
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
output_ids = spsim.sim_jax(sim, gen_spu)(params, input_ids)
print(prompt, tokenizer.decode(output_ids[0]), sep='')

[2026-04-02 16:27:32.834] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-02 16:30:20.110] [info] [api.cc:172] [Profiling] SPU execution gen_spu completed, input processing took 6.312e-05s, execution took 136.377444563s, output processing took 2.13e-06s, total time 136.377509813s.
[2026-04-02 16:30:20.152] [info] [api.cc:220] HLO profiling: total time 136.196992391
[2026-04-02 16:30:20.152] [info] [api.cc:223] - pphlo.exponential, executed 312 times, duration 39.266074632s, send bytes 4023730176 recv bytes 3745916928, send actions 22902, recv actions 21297
[2026-04-02 16:30:20.152] [info] [api.cc:223] - pphlo.convolution, executed 72 times, duration 27.883642609s, send bytes 7421952 recv bytes 7274496, send actions 407, recv actions 398
[2026-04-02 16:30:20.152] [info] [api.cc:223] - pphlo.dot, executed 291 times, duration 26.951928258s, send bytes 25238144 recv bytes 26092416, send actions 785, recv actions 780
[2026-04-02 16:30:20.152] [info] [api.cc:223] 

2′20″后，
```plaintext
Python is a great tool
```

## (3) SPU下验证性能
### 3.1 定义emulator

In [9]:
import sml.utils.emulation as emulation

mode = emulation.Mode.MULTIPROCESS

# note: in MULTIPROCESS mode, bandwidth and latency doesn't work
# emulation.CLUSTER_ABY3_3PC is a hard-coded string
# we copied it to current folder
emulator = emulation.Emulator(
    "3pc.json",
    mode,
    bandwidth=100,
    latency = 10
)

emulator.up()

[2026-04-02 16:30:22,638]-[INFO]-[emulation.py:112]: Start multiprocess cluster...
[2026-04-02 16:30:23,170] [ForkServerProcess-5] Starting grpc server at 127.0.0.1:61924
[2026-04-02 16:30:23,171] [ForkServerProcess-1] Starting grpc server at 127.0.0.1:61920
[2026-04-02 16:30:23,180] [ForkServerProcess-3] Starting grpc server at 127.0.0.1:61922
[2026-04-02 16:30:23,183] [ForkServerProcess-4] Starting grpc server at 127.0.0.1:61923
[2026-04-02 16:30:23,188] [ForkServerProcess-2] Starting grpc server at 127.0.0.1:61921
[2026-04-02 16:30:24,714] [ForkServerProcess-2] Run : builtin_spu_init at node:1
[2026-04-02 16:30:24,714] [ForkServerProcess-3] Run : builtin_spu_init at node:2
[2026-04-02 16:30:24,714] [ForkServerProcess-1] Run : builtin_spu_init at node:0
I0402 16:30:24.723227 3349469     0 external/brpc~/src/brpc/server.cpp:1195] Server[yacl::link::transport::internal::ReceiverServiceImpl] is serving on port=61931.
W0402 16:30:24.723244 3349469     0 external/brpc~/src/brpc/server.cpp

In [10]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
# output_ids = spsim.sim_jax(sim, gen_spu)(
#     model, params, input_ids, n_tokens_to_gen = 10, sample = False, top_k = None, seed = 42
# )
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_spu)(s_params, s_input_ids)

# don't print output here, profiler output may mess up

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
[2026-04-02 16:30:26,241] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-02 16:30:26,306] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-02 16:30:26,310] [ForkServerProcess-4] Run : make_shares at node:3


[2026-04-02 16:30:26.310] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95


[2026-04-02 16:30:32,843] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-02 16:30:32,847] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-02 16:30:32,851] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-02 16:30:32,854] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-02 16:30:32,855] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-02 16:30:32,858] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-02 16:30:32,859] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-02 16:30:32,861] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-02 16:30:32,863] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-02 16:30:32,865] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-02 16:30:32,867] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-02 16:30:32,869] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-02 16:30:32,877] [ForkServerProcess-4

[2026-04-02 16:31:08.700] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-02 16:31:08.701] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-02 16:31:08.702] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-02 16:33:20.142] [info] [api.cc:172] [Profiling] SPU execution gen_spu completed, input processing took 6.602e-05s, execution took 131.807348882s, output processing took 4.22e-06s, total time 131.807419122s.
[2026-04-02 16:33:20.206] [info] [api.cc:220] HLO profiling: total time 131.72138209299996
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.exponential, executed 312 times, duration 42.315413413s, send bytes 4016873472 recv bytes 3750727680, send actions 22888, recv actions 21305
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.convolution, executed 72 times, duration 26.410895758s, send bytes 7643136 recv bytes 7839744, send actions 411, recv actions 418
[2026-04-02 16:33:20.206] [info] [

[2026-04-02 16:33:20,490] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-02 16:33:20,497] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-02 16:33:20,498] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-02 16:33:20,498] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-02 16:33:20,499] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-02 16:33:20,499] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-02 16:33:20,537] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-02 16:33:20,537] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-02 16:33:20,537] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-02 16:33:20,537] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-02 16:33:20,537] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-02 16:33:20,555] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-02 16:33:20,557] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-02 16:33

干掉topk之后速度非常快，现在耗时的大头在exp
```plaintext
[2026-04-02 16:33:20.142] [info] [api.cc:172] [Profiling] SPU execution gen_spu completed, input processing took 6.602e-05s, execution took 131.807348882s, output processing took 4.22e-06s, total time 131.807419122s.
[2026-04-02 16:33:20.206] [info] [api.cc:220] HLO profiling: total time 131.72138209299996
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.exponential, executed 312 times, duration 42.315413413s, send bytes 4016873472 recv bytes 3750727680, send actions 22888, recv actions 21305
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.convolution, executed 72 times, duration 26.410895758s, send bytes 7643136 recv bytes 7839744, send actions 411, recv actions 418
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.dot_general, executed 48 times, duration 22.208245624s, send bytes 81007616 recv bytes 80048384, send actions 98440, recv actions 98528
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.dot, executed 291 times, duration 17.639317426s, send bytes 23644800 recv bytes 22218560, send actions 784, recv actions 763
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.multiply, executed 951 times, duration 6.549204685s, send bytes 282429376 recv bytes 279148616, send actions 2428, recv actions 2314
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.log_plus_one, executed 72 times, duration 3.575090644s, send bytes 380436480 recv bytes 356241408, send actions 3093, recv actions 3155
[2026-04-02 16:33:20.206] [info] [api.cc:223] - pphlo.reciprocal, executed 144 times, duration 3.547714335s, send bytes 304594944 recv bytes 309018624, send actions 8608, recv actions 8755
```

In [11]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

[[ 247 1270 4968]]
Python is a great tool


In [12]:
# emulator.down()